In [69]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import os

df = pd.read_csv('cleaned_data.csv')

# Recreate engineered columns from EDA notebook
df['Ward_Location'] = 'Ward ' + df['Ward'].astype(str) + ' — ' + df['Location']

weather_cols = [col for col in df.columns if col.startswith('Weather_')]
df['Active_Weathers'] = df[weather_cols].apply(
    lambda row: ', '.join([
        col.replace('Weather_', '')
        for col in weather_cols if row[col] == 1
    ]), axis=1
)
df['Active_Weathers'] = df['Active_Weathers'].replace('', 'Unknown')

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape: (913, 35)
Columns: ['Date', 'Month', 'Ward', 'Location', 'Road_Type', 'No_of_vehicles_involved', 'Minor_Injury', 'Severe_Injury', 'Fatal/Death', 'Time_Range', 'Cause', 'Month_Num', 'Nepali_Season', 'Season', 'Weather_Clear', 'Weather_Cold', 'Weather_Cool', 'Weather_Dry', 'Weather_Dusty', 'Weather_Foggy', 'Weather_Frosty', 'Weather_Hot', 'Weather_Humid', 'Weather_Mild', 'Weather_Overcast', 'Weather_Pleasant', 'Weather_Rainy', 'Weather_Thunderstorm', 'Weather_Warm', 'Weather_Windy', 'Severity', 'Total_Injuries', 'Casualties', 'Ward_Location', 'Active_Weathers']


,Date,Month,Ward,Location,Road_Type,No_of_vehicles_involved,Minor_Injury,Severe_Injury,Fatal/Death,Time_Range,...,Weather_Pleasant,Weather_Rainy,Weather_Thunderstorm,Weather_Warm,Weather_Windy,Severity,Total_Injuries,Casualties,Ward_Location,Active_Weathers
0,2081-04-02 00:00:00,Shrawan,4,oil nigam,highway,2,1,0,0,06:00-12:00,...,0,1,0,0,0,low,1,1,Ward 4 — oil nigam,"Foggy, Humid, Overcast, Rainy"
1,2081-04-02 00:00:00,Shrawan,12,roadcess chowk,highway,2,4,0,0,06:00-12:00,...,0,1,0,0,0,low,4,4,Ward 12 — roadcess chowk,"Foggy, Humid, Overcast, Rainy"
2,2081-04-02 00:00:00,Shrawan,4,rajbanshi chowk,highway,2,0,1,0,12:00-18:00,...,0,1,0,0,0,medium,1,1,Ward 4 — rajbanshi chowk,"Foggy, Humid, Overcast, Rainy"
3,2081-04-02 00:00:00,Shrawan,13,rani,inner paved road,1,29,0,1,12:00-18:00,...,0,1,0,0,0,high,29,30,Ward 13 — rani,"Foggy, Humid, Overcast, Rainy"
4,2081-04-03 00:00:00,Shrawan,18,rani,inner paved road,1,1,0,0,18:00-00:00,...,0,1,0,0,0,low,1,1,Ward 18 — rani,"Foggy, Humid, Overcast, Rainy"


In [70]:
drop_cols = [
    'Date',             # redundant
    'Month',            # Month_Num already covers it
    'Season',           # dropping English season — Nepali_Season more relevant
    'Cause',            # excluded from project scope
    'Minor_Injury',     # data leakage
    'Severe_Injury',    # data leakage
    'Fatal/Death',      # data leakage
    'Total_Injuries',   # data leakage
    'Casualties',       # data leakage
    'Active_Weathers',  # text combo — EDA only
]

df_ml = df.drop(columns=drop_cols)

print("Remaining columns:", df_ml.columns.tolist())
print("Shape after drop:", df_ml.shape)
df_ml.head()

Remaining columns: ['Ward', 'Location', 'Road_Type', 'No_of_vehicles_involved', 'Time_Range', 'Month_Num', 'Nepali_Season', 'Weather_Clear', 'Weather_Cold', 'Weather_Cool', 'Weather_Dry', 'Weather_Dusty', 'Weather_Foggy', 'Weather_Frosty', 'Weather_Hot', 'Weather_Humid', 'Weather_Mild', 'Weather_Overcast', 'Weather_Pleasant', 'Weather_Rainy', 'Weather_Thunderstorm', 'Weather_Warm', 'Weather_Windy', 'Severity', 'Ward_Location']
Shape after drop: (913, 25)


,Ward,Location,Road_Type,No_of_vehicles_involved,Time_Range,Month_Num,Nepali_Season,Weather_Clear,Weather_Cold,Weather_Cool,...,Weather_Humid,Weather_Mild,Weather_Overcast,Weather_Pleasant,Weather_Rainy,Weather_Thunderstorm,Weather_Warm,Weather_Windy,Severity,Ward_Location
0,4,oil nigam,highway,2,06:00-12:00,4,Barkha,0,0,0,...,1,0,1,0,1,0,0,0,low,Ward 4 — oil nigam
1,12,roadcess chowk,highway,2,06:00-12:00,4,Barkha,0,0,0,...,1,0,1,0,1,0,0,0,low,Ward 12 — roadcess chowk
2,4,rajbanshi chowk,highway,2,12:00-18:00,4,Barkha,0,0,0,...,1,0,1,0,1,0,0,0,medium,Ward 4 — rajbanshi chowk
3,13,rani,inner paved road,1,12:00-18:00,4,Barkha,0,0,0,...,1,0,1,0,1,0,0,0,high,Ward 13 — rani
4,18,rani,inner paved road,1,18:00-00:00,4,Barkha,0,0,0,...,1,0,1,0,1,0,0,0,low,Ward 18 — rani


In [71]:
print("=== Column Data Types ===\n")
for col in df_ml.columns:
    print(f"{col:30s} → {df_ml[col].dtype} | unique values: {df_ml[col].nunique()}")

=== Column Data Types ===

Ward                           → int64 | unique values: 19
Location                       → str | unique values: 161
Road_Type                      → str | unique values: 3
No_of_vehicles_involved        → int64 | unique values: 4
Time_Range                     → str | unique values: 6
Month_Num                      → int64 | unique values: 12
Nepali_Season                  → str | unique values: 6
Weather_Clear                  → int64 | unique values: 2
Weather_Cold                   → int64 | unique values: 2
Weather_Cool                   → int64 | unique values: 2
Weather_Dry                    → int64 | unique values: 2
Weather_Dusty                  → int64 | unique values: 2
Weather_Foggy                  → int64 | unique values: 2
Weather_Frosty                 → int64 | unique values: 2
Weather_Hot                    → int64 | unique values: 2
Weather_Humid                  → int64 | unique values: 2
Weather_Mild                   → int64 | unique v

In [72]:
# Time Range has natural order — encode manually
time_order_map = {
    '00:00-06:00': 0,
    '06:00-12:00': 1,
    '12:00-18:00': 2,
    '18:00-00:00': 3
}

df_ml['Time_Range'] = df_ml['Time_Range'].map(time_order_map)

print("Time_Range encoded:")
print(df_ml['Time_Range'].value_counts().sort_index())

Time_Range encoded:
Time_Range
0.0     35
1.0    176
2.0    366
3.0    326
Name: count, dtype: int64


In [73]:
print("Null Time_Range rows:", df_ml['Time_Range'].isnull().sum())

# Fill with most common time slot
df_ml['Time_Range'] = df_ml['Time_Range'].fillna(
    df_ml['Time_Range'].mode()[0]
)
print("After fix:", df_ml['Time_Range'].isnull().sum())

Null Time_Range rows: 10
After fix: 0


In [74]:
nepali_season_map = {
    'Basanta': 0,   # Spring
    'Grishma': 1,   # Summer
    'Barkha':  2,   # Monsoon
    'Sharad':  3,   # Autumn
    'Hemanta': 4,   # Early Winter
    'Shishir': 5    # Winter
}

df_ml['Nepali_Season'] = df_ml['Nepali_Season'].map(nepali_season_map)

print("Nepali_Season encoded:")
print(df_ml['Nepali_Season'].value_counts().sort_index())

Nepali_Season encoded:
Nepali_Season
0     70
1     84
2    211
3    202
4    196
5    150
Name: count, dtype: int64


In [75]:
road_type_map = {road: idx for idx, road in 
                 enumerate(df_ml['Road_Type'].unique())}

print("Road_Type mapping:", road_type_map)
df_ml['Road_Type'] = df_ml['Road_Type'].map(road_type_map)

print("\nRoad_Type encoded:")
print(df_ml['Road_Type'].value_counts())

Road_Type mapping: {'highway': 0, 'inner paved road': 1, 'inner unpaved road': 2}

Road_Type encoded:
Road_Type
1    547
0    363
2      3
Name: count, dtype: int64


In [76]:
# Cell 8
df_ml['Ward_Location'] = 'Ward ' + df_ml['Ward'].astype(str) + ' — ' + df_ml['Location']

ward_location_freq = df_ml['Ward_Location'].value_counts(normalize=True)
df_ml['Ward_Location_Risk'] = df_ml['Ward_Location'].map(ward_location_freq)

df_ml = df_ml.drop(columns=['Ward_Location', 'Location'])
print("Ward_Location_Risk sample:")
print(df_ml['Ward_Location_Risk'].describe())

Ward_Location_Risk sample:
count    913.000000
mean       0.022543
std        0.027336
min        0.001095
25%        0.003286
50%        0.009858
75%        0.028478
max        0.084337
Name: Ward_Location_Risk, dtype: float64


In [77]:
# Severity has natural order — encode manually
severity_map = {
    'low':    0,
    'medium': 1,
    'high':   2
}

df_ml['Severity'] = df_ml['Severity'].map(severity_map)

print("Severity encoded:")
print(df_ml['Severity'].value_counts().sort_index())

Severity encoded:
Severity
0    842
1     42
2     29
Name: count, dtype: int64


In [78]:
print("=== Final df_ml Info ===\n")
print(df_ml.info())

print("\n=== Missing Values ===")
print(df_ml.isnull().sum())

print("\n=== Sample Data ===")
df_ml.head()

=== Final df_ml Info ===

<class 'pandas.DataFrame'>
RangeIndex: 913 entries, 0 to 912
Data columns (total 24 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Ward                     913 non-null    int64  
 1   Road_Type                913 non-null    int64  
 2   No_of_vehicles_involved  913 non-null    int64  
 3   Time_Range               913 non-null    float64
 4   Month_Num                913 non-null    int64  
 5   Nepali_Season            913 non-null    int64  
 6   Weather_Clear            913 non-null    int64  
 7   Weather_Cold             913 non-null    int64  
 8   Weather_Cool             913 non-null    int64  
 9   Weather_Dry              913 non-null    int64  
 10  Weather_Dusty            913 non-null    int64  
 11  Weather_Foggy            913 non-null    int64  
 12  Weather_Frosty           913 non-null    int64  
 13  Weather_Hot              913 non-null    int64  
 14  Weather_Hum

,Ward,Road_Type,No_of_vehicles_involved,Time_Range,Month_Num,Nepali_Season,Weather_Clear,Weather_Cold,Weather_Cool,Weather_Dry,...,Weather_Humid,Weather_Mild,Weather_Overcast,Weather_Pleasant,Weather_Rainy,Weather_Thunderstorm,Weather_Warm,Weather_Windy,Severity,Ward_Location_Risk
0,4,0,2,1.0,4,2,0,0,0,0,...,1,0,1,0,1,0,0,0,0,0.035049
1,12,0,2,1.0,4,2,0,0,0,0,...,1,0,1,0,1,0,0,0,0,0.007667
2,4,0,2,2.0,4,2,0,0,0,0,...,1,0,1,0,1,0,0,0,1,0.006572
3,13,1,1,2.0,4,2,0,0,0,0,...,1,0,1,0,1,0,0,0,2,0.074480
4,18,1,1,3.0,4,2,0,0,0,0,...,1,0,1,0,1,0,0,0,0,0.019715


In [79]:
df_ml['Time_Range'].isnull().sum()


np.int64(0)

In [80]:
df_ml.to_csv('../data/processed_data.csv', index=False)
print("✅ processed_data.csv saved successfully!")
print(f"Shape: {df_ml.shape}")
print(f"Columns: {df_ml.columns.tolist()}")

✅ processed_data.csv saved successfully!
Shape: (913, 24)
Columns: ['Ward', 'Road_Type', 'No_of_vehicles_involved', 'Time_Range', 'Month_Num', 'Nepali_Season', 'Weather_Clear', 'Weather_Cold', 'Weather_Cool', 'Weather_Dry', 'Weather_Dusty', 'Weather_Foggy', 'Weather_Frosty', 'Weather_Hot', 'Weather_Humid', 'Weather_Mild', 'Weather_Overcast', 'Weather_Pleasant', 'Weather_Rainy', 'Weather_Thunderstorm', 'Weather_Warm', 'Weather_Windy', 'Severity', 'Ward_Location_Risk']


In [81]:
df_ml.to_csv('../data/processed_data.csv', index=False)
print("✅ processed_data.csv saved successfully!")
print(f"Shape: {df_ml.shape}")
print(f"Columns: {df_ml.columns.tolist()}")

✅ processed_data.csv saved successfully!
Shape: (913, 24)
Columns: ['Ward', 'Road_Type', 'No_of_vehicles_involved', 'Time_Range', 'Month_Num', 'Nepali_Season', 'Weather_Clear', 'Weather_Cold', 'Weather_Cool', 'Weather_Dry', 'Weather_Dusty', 'Weather_Foggy', 'Weather_Frosty', 'Weather_Hot', 'Weather_Humid', 'Weather_Mild', 'Weather_Overcast', 'Weather_Pleasant', 'Weather_Rainy', 'Weather_Thunderstorm', 'Weather_Warm', 'Weather_Windy', 'Severity', 'Ward_Location_Risk']
